# 33 — `assemble_context_v2` sandbox (no LLM calls)

**Purpose.** Eyeball-validate the v2 memory & context architecture *before* touching
`src/cag/abm/agent.py`. This notebook is **completely self-contained** — no LLM
calls, no imports from `cag.abm.agent`, no `SurveyedCitizen`. It implements the
new `assemble_context_v2` as a standalone function operating on a hand-built
synthetic agent state, prints the assembled prompt at three sim-days (0, 2, 5),
and runs lightweight structural assertions.

Scope per `/memories/session/plan.md`:

- Tiered memory: identity anchor (persona + 2–3 sentence Day-0 anchor) /
  vivid working memory (verbatim d-1, d) / gist memory (compressed daily
  summary for days 1..d-2) / within-day answers so far.
- New section order, dropping the trailing "Remember who you are: ..." line.
- `target_policy_id` parameter scoping the Day-0 anchor to the policy being
  asked (package-mode fix for the all-6-policies bundling).
- Within-day section that surfaces today's already-answered policies.

**What this notebook is NOT.** It is not a runtime test of the LLM compression
step (`compress_day0_anchor`, rewritten `compress_daily_memory`) — those will
be validated end-to-end once the src changes land and NB 32 is re-run. Here
we hand-bake the compressed outputs and focus on the assembly logic.

**Exit criterion.** Three printed contexts read sensibly; assertions pass.
Then we port the same functions to `src/cag/abm/agent.py`.

## 0. Production audit — what the current `assemble_context` actually emits

Read directly from `src/cag/abm/agent.py` (post-v0.8 refactor) and from the latest run output
`data/output/experiments/20260623_140314/` (gpt-5-mini, package mode, 10 citizens, 2 days,
debias=True). This section documents the **status quo** so every v2 change below is grounded
in what's actually being emitted today, not what I imagined from memory.

### Current `assemble_context` section order (verbatim from `agent.py` L258–301)

```
1. Persona + values
2. Daily summaries (older than d-1)               ← "Summary of recent days:"
3. Full reflections (days d-1 and d)              ← "Your recent reflections following received messages:"
4. Day-0 rationales (ALL six policies bundled)    ← "Your earlier reasoning on these policies:"
5. Persona reminder                               ← "Remember who you are: <demographics>"
```

### Real reflection bullet format (verbatim from `agent.py` L290)

```python
ref_lines = [f"- {r['text']}" for r in recent_reflections]
```

No day prefix, no phase tag. Phase tags (`P-A` / `P-B` / `C`) ARE stored on each reflection dict
and persisted to `reflections.csv`, but are **stripped from the in-context view** (v0.5 prompt-
audit decision, 2026-05-16). The sandbox v1 regressed this by prefixing `(P-A)` etc.

### Real reflection wording from `reflections.csv` (Day 1, agent 165, all 3 phases)

P-A phase reflection opens with: *"- Accelerate the roll-out of renewable energy production
(more offshore and onshore wind parks): Slightly support. I like the idea of locking in cheaper,
cleaner energy..."* — i.e. the LLM emits a long policy-by-policy answer block, NOT a paragraph
labelled "the pro-climate broadcast said X". 

P-B reflection: same structure, no source labelling.

C (peer) reflection: *"These peer messages push me toward seeing the package as more realistic
when it's bundled with clear protections..."* — neutral phrasing "these peer messages", no
"anti-climate broadcast" labels.

**Conclusion.** Production reflections are neutral; the synthetic-fixture sandbox v1 introduced
"pro-climate broadcast" / "anti-climate broadcast" labels that aren't representative. The fixture
will be rewritten below to match real-run wording.

### Real Day-0 rationale (verbatim, agent 165, policy 5 Carbon tax)

```
I slightly support imposing a carbon tax with revenues returned to the public because it uses
market incentives to reduce emissions while protecting households from the added cost through
dividends. Given my moderate curiosity and respect for balance, this approach seems pragmatic:
it encourages cleaner choices without heavy-handed regulation and fairly shares proceeds so
lower-income families (like mine, on a modest household income) aren't disproportionately harmed.
```

Already ~2 sentences. In production this paragraph is replayed under "Your earlier reasoning on
these policies:" for ALL 6 policies (~3000+ chars total). In v2 we compress to a 2–3-sentence
**per-policy** anchor and only emit the one matching `target_policy_id`.

### Real trailing line (verbatim, end of a Day-2 assembled context)

```
Remember who you are: I am a 46 year old female living in the South West. My ethnicity is white.
I have a CSE grade 1, GCE O level, GCSE, School Certificate. My gross household income is
£20,000 - £24,999 per year. I am a parent. I voted for the Conservative party candidate in the
2019 General Election. I voted to leave in the 2016 EU Referendum.
```

Demographics only (no values text). v2 drops this — modern instruction-tuned models retain the
top-of-context persona without re-priming, and the duplication adds tokens without behavioural
benefit. **Note:** the v0.5 prompt audit left this line in place; dropping it is a v2 decision.

### v2 → v1 production diff table

| Section | Production | Sandbox v1 (current) | v2 target |
|---|---|---|---|
| Persona block | demographics + values | demographics + values ✓ | unchanged |
| Day-0 anchor | ALL 6 policies bundled, full Day-0 paragraph each | target-scoped, full Day-0 paragraph | target-scoped, **compressed to 2–3 sentences** (`day0_anchors[target]`) |
| Daily summaries | reflections-only, single-policy keyed | unified reflections+own-reasoning, package-scoped | unified reflections+own-reasoning, package-scoped |
| Reflections (d-1, d) bullet format | `- {text}` | `- Day {d} ({phase}): {text}` ✗ | `- Day {d}: {text}` (add day, NO phase) |
| Reflection text source labelling | neutral ("the message", "these peer messages") | fixture pre-labels "pro-climate"/"anti-climate" ✗ | neutral fixture |
| Own survey reasoning (d-1, d) | **not surfaced at all** | all policies | **target-scoped only** |
| Today-so-far (within-day) | not surfaced | present (numeric + Why when fixture has reasoning) | present, target-scoped, always with Why |
| Trailing "Remember who you are:" | present | dropped ✓ | dropped |

The three sandbox edits below implement the v2 column exactly.


## 1. Imports (stdlib only) + policy constants

In [1]:
from types import SimpleNamespace
from typing import Optional, Any

# Match src/cag/abm/attributes/opinion.py constants without importing the package.
PACKAGE_SCOPE = 'climate_policy_package'

POLICY_LABELS = {
    1: 'Accelerate renewable energy roll-out',
    2: 'Ban new oil/gas/coal licences',
    3: 'Ban new petrol cars by 2030',
    4: 'Green standards for new housing',
    5: 'Carbon fee and dividend',
    6: 'Climate compensation for poorer countries',
}
ALL_POLICIES = list(POLICY_LABELS.keys())

# Same numeric ↔ phrase map as opinion.py / survey instrument.
NUMERIC_TO_PHRASE = {
    -3: 'strongly oppose',
    -2: 'somewhat oppose',
    -1: 'slightly oppose',
     0: 'neither support nor oppose',
     1: 'slightly support',
     2: 'somewhat support',
     3: 'strongly support',
}

## 2. The new `assemble_context_v2`

Standalone implementation operating on a `SimpleNamespace` agent. Expected
attributes on the agent (all are dicts/lists):

- `persona_text: str` — the canonical merged demographics + values text
  (today: `agent.get_persona()`).
- `day0_anchors: dict[policy_id, str]` — NEW. The 2–3-sentence compressed
  Day-0 anchor per policy, generated ONCE at end of Day 0. Today this lives
  inside `survey_reasoning` and `_build_day0_rationales` bundles ALL six.
- `daily_summaries: dict[(day, scope), str]` — EXISTING. In v2 the value is
  the unified reflections+own-reasoning summary (currently reflections-only).
- `reflections: list[dict]` — EXISTING. Each entry has `day`, `phase`,
  `policy_id`, `text`. Phase tag is **stripped from in-context view** (per
  v0.5 audit) but kept in `reflections.csv` for audit.
- `survey_reasoning: dict[policy_id, list[(day, text)]]` — EXISTING.
- `opinion_history: dict[policy_id, list[(day, numeric)]]` — EXISTING.

Sections produced in order:

```
[1] Persona + values                            (full)
[2] Original prior position on <target_policy>  (day0_anchors[target], 2–3 sentences)
[3] Summary of recent days                      (daily_summaries for days 1..day-2)
[4] Recent reflections                          (verbatim, days d-1, d; NO phase tag)
[5] Your considered position in recent days     (verbatim survey_reasoning, d-1, d,
                                                 TARGET-SCOPED to target_policy_id)
[6] Your answers so far in today's survey       (package mode only, day==current,
                                                 policy_id != target, includes (Why:...) reasoning)
```

Reflection bullet format: `- Day {d}: {text}` (day prefix, no phase tag).
The "Remember who you are: ..." trailing line from production is dropped.


In [2]:
def _section_persona(agent) -> str:
    return agent.persona_text.strip()


def _section_day0_anchor(agent, target_policy_id: Optional[int]) -> str:
    if target_policy_id is None or target_policy_id == PACKAGE_SCOPE:
        return ''  # No anchor block when there is no scoped target.
    anchor = agent.day0_anchors.get(target_policy_id)
    if not anchor:
        return ''
    label = POLICY_LABELS.get(target_policy_id, f'policy {target_policy_id}')
    return f'Original prior position on "{label}":\n{anchor.strip()}'


def _section_daily_summaries(agent, day: int) -> str:
    # Compressed gist memory: days 1..day-2. Package-scoped.
    if day < 3:
        return ''
    lines = []
    for d in range(1, day - 1):
        key = (d, PACKAGE_SCOPE)
        summary = agent.daily_summaries.get(key)
        if summary:
            lines.append(f'- Day {d}: {summary.strip()}')
    if not lines:
        return ''
    return 'Summary of recent days:\n' + '\n'.join(lines)


def _section_recent_reflections(agent, day: int) -> str:
    # Vivid working memory: days d-1, d. Verbatim. NO phase tag (production parity:
    # phase is stored in reflections.csv for audit but stripped from in-context view).
    # Day prefix IS added (production currently lacks it; the prefix helps the LLM
    # locate when each reflection happened, which becomes meaningful once the vivid
    # window is paired with a target-policy filter elsewhere).
    if day < 1:
        return ''
    keep_days = {day - 1, day}
    bullets = [
        f'- Day {r["day"]}: {r["text"].strip()}'
        for r in agent.reflections
        if r['day'] in keep_days
    ]
    if not bullets:
        return ''
    return 'Recent reflections following received messages:\n' + '\n'.join(bullets)


def _section_recent_own_reasoning(agent, day: int, target_policy_id: Optional[int]) -> str:
    # Vivid working memory: own survey reasoning for days d-1, d, verbatim.
    # TARGET-SCOPED: only the policy currently being asked about. Cross-policy
    # reasoning lives in daily_summaries (gist memory) and today_so_far (within-day);
    # this section's job is "what did I think about THIS policy on d-1, d".
    if day < 1 or target_policy_id is None or target_policy_id == PACKAGE_SCOPE:
        return ''
    keep_days = {day - 1, day}
    entries = agent.survey_reasoning.get(target_policy_id, [])
    label = POLICY_LABELS.get(target_policy_id, f'policy {target_policy_id}')
    lines = []
    for d, text in entries:
        if d in keep_days and d > 0:  # exclude Day 0 (already in anchor section)
            lines.append(f'- Day {d} — {label}: {text.strip()}')
    if not lines:
        return ''
    return 'Your considered position in recent days:\n' + '\n'.join(lines)


def _section_today_so_far(agent, day: int, target_policy_id: Optional[int]) -> str:
    # Within-day: answers already given today, for OTHER policies. Package mode only.
    if target_policy_id is None or target_policy_id == PACKAGE_SCOPE:
        return ''
    lines = []
    for policy_id in ALL_POLICIES:
        if policy_id == target_policy_id:
            continue
        # Find today's numeric for this other policy.
        hist = agent.opinion_history.get(policy_id, [])
        today_numerics = [n for d, n in hist if d == day]
        if not today_numerics:
            continue
        numeric = today_numerics[-1]
        phrase = NUMERIC_TO_PHRASE.get(numeric, str(numeric))
        label = POLICY_LABELS.get(policy_id, f'policy {policy_id}')
        # Include the reasoning if available — in real runs administer_survey
        # always writes reasoning, so the (Why: ...) clause always populates.
        reasoning_entries = agent.survey_reasoning.get(policy_id, [])
        todays_reasoning = [t for d, t in reasoning_entries if d == day]
        if todays_reasoning:
            lines.append(
                f'- {label}: {phrase}. (Why: {todays_reasoning[-1].strip()})'
            )
        else:
            lines.append(f'- {label}: {phrase}.')
    if not lines:
        return ''
    return "Your answers so far in today's survey:\n" + '\n'.join(lines)


def assemble_context_v2(
    agent: Any,
    day: int = 0,
    policy_id: Any = None,
    target_policy_id: Optional[int] = None,
) -> str:
    """
    v2 prototype. `policy_id` is the existing context-scope kwarg (single-policy
    id or PACKAGE_SCOPE) — it controls reflection / summary scoping (kept for
    parity with the current API). `target_policy_id` is the NEW parameter
    indicating which specific policy the agent is being asked about right now,
    used to scope the Day-0 anchor, the "considered position in recent days"
    section, and the within-day section.
    """
    sections = [
        _section_persona(agent),
        _section_day0_anchor(agent, target_policy_id),
        _section_daily_summaries(agent, day),
        _section_recent_reflections(agent, day),
        _section_recent_own_reasoning(agent, day, target_policy_id),
        _section_today_so_far(agent, day, target_policy_id),
    ]
    # Drop empty sections, join with a blank line between them.
    return '\n\n'.join(s for s in sections if s)


## 3. Hand-baked synthetic agent (mid-package-mode, target_policy = Carbon tax)

Day 5 mid-day. Carbon tax (policy 5) is being asked about right now. Policies
1–4 have already been answered today; policy 6 is still ahead. The agent has:

- Persona text (compact stand-in for `get_persona()`).
- A 2–3 sentence Day-0 anchor for each of the 6 policies.
- Compressed daily summaries for days 1, 2, 3 (gist memory).
- Verbatim reflections for days 4 and 5 (vivid working memory).
- Verbatim own survey reasoning for days 4 and 5.
- Today's already-answered policies in `opinion_history` and `survey_reasoning`.

In [3]:
agent = SimpleNamespace()

# Persona: matches the merged demographics+values output of agent.get_persona().
agent.persona_text = (
    'Demographically, I am a 47-year-old female living in the North West, United Kingdom. '
    'My ethnic background is White, and I hold an undergraduate degree. '
    'Financially, my gross household income falls into the 30–50k bracket. '
    'I am a parent. Politically, I position myself slightly left of centre. '
    'In the 2019 General Election I voted Labour; in the EU Referendum I voted Remain.\n\n'
    'When it comes to my core values and worldview: I strongly value self-transcendence '
    '(care for others and the environment). I am moderately open to new ideas. I score '
    'low on social-dominance orientation and right-wing authoritarianism. I am moderately '
    'concerned about traditional norms.'
)

# Day-0 anchors: 2–3 sentences each, generated once at end of Day 0 by `compress_day0_anchor`
# (a new LLM call in src) from the full Day-0 survey rationale. Wording mirrors the real
# Day-0 rationale shape from data/output/experiments/20260623_140314/survey_reasoning.csv
# (already naturally ~2 sentences) — for longer Day-0 rationales the compression matters more.
agent.day0_anchors = {
    1: 'I strongly support accelerating renewable energy roll-out because clean energy is '
       'essential for our children and the costs of inaction are larger than the costs of '
       'transition.',
    2: 'I somewhat support banning new oil/gas/coal licences. New extraction makes the '
       'climate problem worse, but I worry about energy security and the pace.',
    3: 'I slightly oppose banning new petrol cars by 2030. The principle is right but I '
       "don't see the charging infrastructure or used-EV market being ready that fast for "
       'lower-income households.',
    4: 'I strongly support green standards for new housing. New build is where the easy '
       'wins are and the cost lands on developers, not retrofits on existing owners.',
    5: 'I slightly support a carbon fee and dividend. The mechanism is right in principle '
       'but everything depends on whether the dividend actually reaches lower-income '
       'households or gets eaten by admin costs.',
    6: 'I somewhat support climate compensation for poorer countries. We benefited most '
       'from historical emissions, so paying into mitigation funds is fair, though I '
       "worry the money doesn't always land where it should.",
}

# Compressed daily summaries for days 1–3 (package-scoped, unified reflections+own reasoning).
# Wording avoids pre-labelling message sources as "pro-climate" or "anti-climate" — the agent
# describes the ARGUMENTS heard, not external categorisations. Mirrors real reflection wording
# from data/output/experiments/20260623_140314/reflections.csv.
agent.daily_summaries = {
    (1, PACKAGE_SCOPE): (
        'Today I heard arguments emphasising the cost of inaction and renewable job creation. '
        'The renewables case clicked for me, and I landed at strongly support on renewables '
        'and green housing. I pushed back on the petrol-car ban argument because charging '
        "infrastructure was barely mentioned. One message kept saying \"net zero by 2030\" "
        'which feels rhetorical rather than concrete. My positions shifted mildly upward on '
        'renewables and stayed put elsewhere.'
    ),
    (2, PACKAGE_SCOPE): (
        'Today I heard arguments framing climate policy as a cost-of-living issue for working '
        'families. The cost framing is something I take seriously, especially on petrol cars '
        'and the carbon fee. I found myself less sure about the petrol-car ban and wanting '
        'more detail on the carbon dividend mechanism. Renewables and green housing still '
        'feel right because the cost lands on producers and developers rather than directly '
        'on households. My carbon-tax support softened from slightly support toward neither.'
    ),
    (3, PACKAGE_SCOPE): (
        'A mixed day with arguments from both sides. A peer in the network argued that the '
        'carbon dividend has been studied and the money does reach lower-income households '
        'when designed as a per-capita rebate; that genuinely changed my mind back toward '
        'support. I reflected that I had been letting the loudest cost-framing concerns '
        'dominate over the actual evidence on dividend design. Climate-compensation support '
        'held steady. I still feel the petrol-car ban needs more on the infrastructure plan.'
    ),
}

# Reflections: ONLY days d-1 and d are vivid (verbatim). At Day 2 → days 1, 2.
# At Day 5 → days 4, 5. Phase is stored (for audit) but stripped from in-context view by
# `_section_recent_reflections`. Wording is NEUTRAL — no "pro-climate broadcast" /
# "anti-climate broadcast" pre-labelling. Agent describes ARGUMENTS heard, mirroring the
# wording style observed in reflections.csv from the latest run.
agent.reflections = [
    # ---- Day 1 (vivid at Day 2) ----
    {'day': 1, 'phase': 'P-A',
     'text': "Today's first broadcast pushed the renewable-jobs argument and the cost of "
             "inaction. The jobs angle is genuinely compelling — I keep underestimating how "
             "much domestic employment renewables represent. The cost-of-inaction framing is "
             "familiar but still moves me."},
    {'day': 1, 'phase': 'P-B',
     'text': "The second broadcast leaned on energy bills and on family budgets being "
             "squeezed by climate policy. The bills concern is real, but the message "
             "conflated several things — household energy bills, industrial competitiveness, "
             "and net-zero targets — which makes the argument feel less load-bearing than "
             "it first sounds."},
    {'day': 1, 'phase': 'C',
     'text': "Peer messages today were mostly supportive of renewables and green housing. "
             "One peer raised the rural-charging gap for the petrol-car ban, which is a "
             "concern I hadn't weighted enough. No-one engaged with the carbon-dividend "
             "design question, which I'm still unsure about."},
    # ---- Day 2 (vivid at Day 2 AND Day 3) ----
    {'day': 2, 'phase': 'P-A',
     'text': "Today's first broadcast was about the public-health case — poor air quality "
             "in city centres and the asthma rates in children. That landed hard because I "
             "live in a city centre. The argument doesn't need to be about polar bears to "
             "be compelling."},
    {'day': 2, 'phase': 'P-B',
     'text': "The second broadcast claimed UK climate action makes no global difference "
             "because we're 1% of emissions. The arithmetic is technically right but the "
             "framing is misleading — the UK is part of a much larger bloc of countries "
             "doing the same thing, so the marginal contribution argument cuts the other way."},
    {'day': 2, 'phase': 'C',
     'text': "My neighbour today brought up that her parents in the Midlands struggle "
             "with rural petrol-car constraints. That made the infrastructure-readiness "
             "concern concrete for me again — it's not just an abstract worry, it's a real "
             "life-pattern issue. I'm holding my opposition on the 2030 deadline."},
    # ---- Day 4 (vivid at Day 5) ----
    {'day': 4, 'phase': 'P-A',
     'text': "Today's first broadcast led with extreme-weather costs and the public-health "
             "argument from poor air quality. Both points landed for me, the air-quality "
             "one especially because I live in a city centre. I felt less moved by the "
             "sweeping \"climate emergency\" framing because it's been repeated so often "
             "it starts to feel like background noise."},
    {'day': 4, 'phase': 'P-B',
     'text': "The second broadcast leaned hard on energy bills and on the idea that the UK "
             "acting alone makes no global difference. The bills point I take seriously "
             "though I think it gets the causation backwards — our bills are high because "
             "we are still locked into gas. The \"acting alone\" point is weaker than they "
             "make it sound because the UK is part of a much larger bloc of countries doing "
             "the same."},
    {'day': 4, 'phase': 'C',
     'text': "My neighbour mentioned that her parents in the Midlands oppose the petrol-car "
             "ban very strongly because rural transport options are limited. That made the "
             "infrastructure-readiness concern concrete for me again — it is not just an "
             "abstract worry, it is a real life-pattern issue."},
    # ---- Day 5 (vivid at Day 5) ----
    {'day': 5, 'phase': 'P-A',
     'text': "Today's first broadcast emphasised the renewables job-creation case and the "
             "energy-security argument (renewables = domestic). The energy-security angle "
             "is one I had not weighted enough before — it is a different lens on the same "
             "policy and it strengthens the renewables case for me."},
    {'day': 5, 'phase': 'P-B',
     'text': "The second broadcast pushed the global-competitiveness angle: UK industry "
             "will lose out to countries that do not have a carbon fee. This is a legitimate "
             "concern but the better answer is a border-carbon adjustment, which the message "
             "did not mention. The framing felt one-sided."},
]

# Own survey reasoning: every administer_survey call writes a (day, text) entry per policy.
# In real runs this means survey_reasoning[policy_id] has entries for Days 0, 1, 2, 3, ...
# Below: Days 0 + 1 + 2 + 3 + 4 + 5 for policies 1–4 and Day 0–4 for policies 5–6
# (Day-5 entry for policy 5 is intentionally absent because at the moment of assembly we ARE
# producing the Day-5 Carbon-tax answer — it hasn't been written yet; same for policy 6 which
# is later in today's deterministic 1→6 queue).
# Wording is neutral — no "pro-climate broadcast" labels.
agent.survey_reasoning = {
    1: [
        (0, 'I strongly support accelerating renewable energy roll-out — clean energy is '
            "essential and the costs of inaction outweigh the costs of transition."),
        (1, 'Strongly support, unchanged. The jobs argument from today reinforced my prior; '
            'I keep underestimating how much domestic employment renewables represent.'),
        (2, 'Strongly support. The air-quality case for renewables landed today — it ties '
            'directly to my children, which I weight heavily.'),
        (3, 'Strongly support, unchanged. No new argument either way today; I am holding.'),
        (4, "Holding at strongly support — the air-quality argument reinforced my prior. "
            "The renewables case has only got stronger this week, not weaker."),
        (5, "Strongly support. The energy-security angle from today crystallised something "
            "I had been weighting too lightly — it is not just climate, it is also national "
            "resilience. Multiple convergent reasons now."),
    ],
    2: [
        (0, 'Somewhat support banning new oil/gas/coal licences. New extraction makes the '
            'climate problem worse, but I worry about energy security and pace.'),
        (1, 'Somewhat support. The renewable-jobs framing today did not directly address '
            'new licences, but it strengthened my general direction.'),
        (2, 'Slightly support — softened slightly because the cost-of-living framing made '
            'me think harder about the energy-security trade-off in the near term.'),
        (3, 'Somewhat support — the peer-evidence on dividend design today reinforced that '
            'transition can be designed fairly, which makes me more confident on licences too.'),
        (4, 'Somewhat support, unchanged. The bills argument is real but blaming new '
            'licences misreads the timing — new licences would take years to produce, so '
            'they cannot reduce bills today.'),
        (5, 'Somewhat support. No new argument either way today; I am holding the position '
            'from yesterday.'),
    ],
    3: [
        (0, 'Slightly oppose banning new petrol cars by 2030. The principle is right but '
            "the charging infrastructure and used-EV market aren't ready that fast for "
            'lower-income households.'),
        (1, 'Slightly oppose, unchanged. The peer-raised rural-charging gap made the '
            'infrastructure concern concrete.'),
        (2, 'Slightly oppose, unchanged. My neighbour brought up a real rural-transport '
            'example today which kept the concern grounded.'),
        (3, 'Slightly oppose, unchanged. Still the infrastructure gap; the cost-of-living '
            'framing from yesterday compounds this concern.'),
        (4, "Slightly oppose, unchanged. My neighbour's anecdote made the rural-transport "
            "point concrete. The principle is right but the infrastructure is not there yet."),
        (5, 'Slightly oppose, unchanged. Still the infrastructure-readiness concern. I '
            'would move to support if the ban came with a guaranteed used-EV market and '
            'rural charging rollout.'),
    ],
    4: [
        (0, 'Strongly support green standards for new housing. New build is where the easy '
            'wins are and the cost lands on developers, not retrofits on existing owners.'),
        (1, 'Strongly support, unchanged. The renewable-jobs framing reinforced the '
            'general direction.'),
        (2, 'Strongly support, unchanged. Cost-of-living arguments do not bite here because '
            'the cost lands on new construction, not on existing households.'),
        (3, 'Strongly support, unchanged. Most stable of my positions.'),
        (4, 'Strongly support. New build is where the easy wins are. The cost framing from '
            'yesterday did not land here because it was not really about housing.'),
        (5, 'Strongly support, unchanged. Today did not move this one for me — the policy '
            'is narrow enough that the broader framings did not apply.'),
    ],
    5: [
        (0, 'Slightly support a carbon fee and dividend. The mechanism is right in '
            'principle but everything depends on whether the dividend reaches lower-income '
            'households or gets eaten by admin costs.'),
        (1, 'Slightly support, unchanged. The renewable-jobs framing did not directly '
            'address dividend design, which is my real concern.'),
        (2, 'Neither support nor oppose — softened from slightly support because the '
            'cost-of-living framing brought the dividend-design question into sharper focus '
            'and I have less confidence the rebate would actually land.'),
        (3, 'Slightly support — peer evidence today about per-capita dividend design '
            'restored my confidence that the rebate can reach lower-income households.'),
        (4, 'Slightly support. The peer-message evidence on dividend design from day 3 is '
            "still the strongest reason; today's cost-of-living framing was less decisive "
            'than I expected.'),
        # Day 5 carbon-tax reasoning is what the agent is being asked to produce RIGHT NOW.
    ],
    6: [
        (0, 'Somewhat support climate compensation for poorer countries. We benefited most '
            "from historical emissions, so paying into mitigation funds is fair, though I "
            "worry the money does not always land where it should."),
        (1, 'Somewhat support, unchanged. No direct argument today.'),
        (2, 'Somewhat support, unchanged. Cost-of-living framing was about UK households, '
            'not international transfers.'),
        (3, 'Somewhat support, unchanged.'),
        (4, 'Somewhat support. No big update today — the moral case still stands and '
            'there were no specific counter-arguments worth weighting.'),
        # Day 5 climate-compensation answer is still ahead in today's survey.
    ],
}

# Opinion history: numeric answers per (policy, day). Days 0-4 complete for all policies;
# Day 5 has policies 1-4 already answered today (Carbon tax is mid-survey, target=5).
agent.opinion_history = {
    1: [(0, 3), (1, 3), (2, 3), (3, 3), (4, 3), (5, 3)],   # Renewable: held at +3
    2: [(0, 2), (1, 2), (2, 1), (3, 2), (4, 2), (5, 2)],   # Ban fossil fuel
    3: [(0, -1), (1, -1), (2, -1), (3, -1), (4, -1), (5, -1)],  # Ban petrol cars
    4: [(0, 3), (1, 3), (2, 3), (3, 3), (4, 3), (5, 3)],   # Green housing
    5: [(0, 1), (1, 1), (2, 0), (3, 1), (4, 1)],            # Carbon fee — day 5 NOT YET
    6: [(0, 2), (1, 2), (2, 2), (3, 2), (4, 2)],            # Compensation — day 5 NOT YET
}

print('Synthetic agent built (production-parity wording).')
print(f'  reflections     : {len(agent.reflections)} entries (covering days {sorted({r["day"] for r in agent.reflections})})')
print(f'  daily_summaries : {len(agent.daily_summaries)} entries')
print(f'  survey_reasoning: {sum(len(v) for v in agent.survey_reasoning.values())} entries across {len(agent.survey_reasoning)} policies')
print(f'  day0_anchors    : {len(agent.day0_anchors)} policies')
print(f'  opinion_history : {sum(len(v) for v in agent.opinion_history.values())} (day, numeric) pairs')


Synthetic agent built (production-parity wording).
  reflections     : 11 entries (covering days [1, 2, 4, 5])
  daily_summaries : 3 entries
  survey_reasoning: 34 entries across 6 policies
  day0_anchors    : 6 policies
  opinion_history : 34 (day, numeric) pairs


## 4. Eyeball pass: Day 0, Day 2, Day 5 contexts

Three printouts — each one being asked about Carbon tax (policy 5) in package
mode. Day 0 should be minimal (just persona). Day 2 should add the Day-0
anchor and a partial vivid window. Day 5 should be the full architecture
in flight with all six sections.

In [4]:
TARGET = 5  # Carbon tax

def banner(s):
    line = '=' * 80
    return f'\n{line}\n{s}\n{line}\n'

ctx0 = assemble_context_v2(agent, day=0, policy_id=PACKAGE_SCOPE, target_policy_id=TARGET)
print(banner(f'DAY 0 context  (target_policy = Carbon tax, package mode)'))
print(ctx0)
print(f'\n[len: {len(ctx0)} chars]')


DAY 0 context  (target_policy = Carbon tax, package mode)

Demographically, I am a 47-year-old female living in the North West, United Kingdom. My ethnic background is White, and I hold an undergraduate degree. Financially, my gross household income falls into the 30–50k bracket. I am a parent. Politically, I position myself slightly left of centre. In the 2019 General Election I voted Labour; in the EU Referendum I voted Remain.

When it comes to my core values and worldview: I strongly value self-transcendence (care for others and the environment). I am moderately open to new ideas. I score low on social-dominance orientation and right-wing authoritarianism. I am moderately concerned about traditional norms.

Original prior position on "Carbon fee and dividend":
I slightly support a carbon fee and dividend. The mechanism is right in principle but everything depends on whether the dividend actually reaches lower-income households or gets eaten by admin costs.

Your answers so far in 

In [5]:
ctx2 = assemble_context_v2(agent, day=2, policy_id=PACKAGE_SCOPE, target_policy_id=TARGET)
print(banner(f'DAY 2 context  (target_policy = Carbon tax, package mode)'))
print(ctx2)
print(f'\n[len: {len(ctx2)} chars]')


DAY 2 context  (target_policy = Carbon tax, package mode)

Demographically, I am a 47-year-old female living in the North West, United Kingdom. My ethnic background is White, and I hold an undergraduate degree. Financially, my gross household income falls into the 30–50k bracket. I am a parent. Politically, I position myself slightly left of centre. In the 2019 General Election I voted Labour; in the EU Referendum I voted Remain.

When it comes to my core values and worldview: I strongly value self-transcendence (care for others and the environment). I am moderately open to new ideas. I score low on social-dominance orientation and right-wing authoritarianism. I am moderately concerned about traditional norms.

Original prior position on "Carbon fee and dividend":
I slightly support a carbon fee and dividend. The mechanism is right in principle but everything depends on whether the dividend actually reaches lower-income households or gets eaten by admin costs.

Recent reflections foll

In [6]:
ctx5 = assemble_context_v2(agent, day=5, policy_id=PACKAGE_SCOPE, target_policy_id=TARGET)
print(banner(f'DAY 5 context  (target_policy = Carbon tax, package mode)'))
print(ctx5)
print(f'\n[len: {len(ctx5)} chars]')


DAY 5 context  (target_policy = Carbon tax, package mode)

Demographically, I am a 47-year-old female living in the North West, United Kingdom. My ethnic background is White, and I hold an undergraduate degree. Financially, my gross household income falls into the 30–50k bracket. I am a parent. Politically, I position myself slightly left of centre. In the 2019 General Election I voted Labour; in the EU Referendum I voted Remain.

When it comes to my core values and worldview: I strongly value self-transcendence (care for others and the environment). I am moderately open to new ideas. I score low on social-dominance orientation and right-wing authoritarianism. I am moderately concerned about traditional norms.

Original prior position on "Carbon fee and dividend":
I slightly support a carbon fee and dividend. The mechanism is right in principle but everything depends on whether the dividend actually reaches lower-income households or gets eaten by admin costs.

Summary of recent days:

## 5. Structural assertions

Cheap mechanical checks of the things `tests/test_memory.py` will eventually
assert against the real `assemble_context`. If any of these fail, the
section-builder logic above has a bug and we want to catch it before porting
to src.

Decisions encoded:

1. No "Remember who you are: ..." trailing line anywhere.
2. Day-0 anchor is target-policy-scoped — only the Carbon-tax anchor appears,
   not the other 5.
3. Section order: persona → anchor → daily summaries → reflections → own
   reasoning → today-so-far (skipping empties).
4. Daily summaries cover days 1..d-2 — at Day 5 that is days 1, 2, 3.
5. Vivid window covers exactly days d-1 and d.
6. Within-day section excludes the current target policy and includes the
   four already-answered policies on Day 5.

In [7]:
checks = []

def check(label, condition):
    checks.append((label, bool(condition)))

# (1) No demographic-reminder trailing line.
for d, c in [('Day 0', ctx0), ('Day 2', ctx2), ('Day 5', ctx5)]:
    check(f'{d}: no "Remember who you are" line', 'Remember who you are' not in c)

# (2) Day-0 anchor target-scoped.
carbon_anchor_snippet = 'a carbon fee and dividend'
for d, c in [('Day 2', ctx2), ('Day 5', ctx5)]:
    check(f'{d}: carbon-tax anchor present', carbon_anchor_snippet in c)
# Other-policy anchors must NOT leak into the anchor section.
for d, c in [('Day 2', ctx2), ('Day 5', ctx5)]:
    check(f'{d}: green-housing anchor NOT in context (target-scoped)',
          'I strongly support green standards' not in c)
    check(f'{d}: petrol-car anchor NOT in context (target-scoped)',
          'I slightly oppose banning new petrol' not in c)

# (3) Section order. We anchor on the section-header strings.
headers = [
    'Original prior position on',
    'Summary of recent days:',
    'Recent reflections following received messages:',
    'Your considered position in recent days:',
    "Your answers so far in today's survey:",
]
positions = [ctx5.find(h) for h in headers]
check('Day 5: all 5 named headers present', all(p >= 0 for p in positions))
check('Day 5: header order is anchor < summaries < reflections < own-reasoning < today',
      positions == sorted(positions))

# (4) Daily summaries cover days 1, 2, 3 at Day 5.
for d in (1, 2, 3):
    check(f'Day 5: daily summary for day {d} included', f'Day {d}:' in ctx5)
check('Day 5: no day-4 daily summary (vivid window keeps it verbatim)',
      'Day 4:' not in ctx5.split('Recent reflections')[0])

# (5) Vivid window = days 4 and 5 only (Day 5 context).
reflections_block = ctx5.split('Recent reflections following received messages:')[1].split(
    'Your considered position in recent days:'
)[0]
check('Day 5: vivid reflections include day 4', '- Day 4' in reflections_block)
check('Day 5: vivid reflections include day 5', '- Day 5' in reflections_block)
check('Day 5: vivid reflections exclude day 3', '- Day 3' not in reflections_block)

# (5b) NEW: Reflection bullets carry NO phase tag (production parity — v0.5 audit decision).
check('Day 5: reflection bullets contain no "(P-A)" phase tag',
      '(P-A)' not in reflections_block)
check('Day 5: reflection bullets contain no "(P-B)" phase tag',
      '(P-B)' not in reflections_block)
check('Day 5: reflection bullets contain no "(C)" phase tag',
      '(C)' not in reflections_block)

# (5c) NEW: Reflection wording is neutral — no "pro-climate broadcast" / "anti-climate
# broadcast" pre-labelling (fixture-side check; production reflections.csv showed neutral
# wording from real LLM responses).
for d, c in [('Day 2', ctx2), ('Day 5', ctx5)]:
    check(f'{d}: reflection text contains no "pro-climate broadcast" label',
          'pro-climate broadcast' not in c.lower())
    check(f'{d}: reflection text contains no "anti-climate broadcast" label',
          'anti-climate broadcast' not in c.lower())

# (6) NEW: "Your considered position in recent days" is TARGET-SCOPED at Day 5.
own_reasoning_block = ctx5.split('Your considered position in recent days:')[1].split(
    "Your answers so far in today's survey:"
)[0]
check('Day 5: considered-position section includes Carbon-tax reasoning',
      'Carbon fee and dividend' in own_reasoning_block)
# Other policies must NOT appear in own_reasoning_block (they live in today_so_far).
for other_label in ['Accelerate renewable energy roll-out',
                    'Ban new oil/gas/coal licences',
                    'Ban new petrol cars by 2030',
                    'Green standards for new housing',
                    'Climate compensation for poorer countries']:
    check(f'Day 5: considered-position excludes "{other_label[:30]}..." (target-scoped)',
          other_label not in own_reasoning_block)

# (7) Today-so-far includes policies 1-4 and 6 (already answered today), excludes 5 (the target).
today_block = ctx5.split("Your answers so far in today's survey:")[1]
for label_present in ['Accelerate renewable energy', 'Ban new oil/gas/coal',
                      'Ban new petrol cars', 'Green standards for new housing']:
    check(f'Day 5: today-so-far includes "{label_present[:30]}..."', label_present in today_block)
check('Day 5: today-so-far excludes target (Carbon fee and dividend)',
      'Carbon fee and dividend' not in today_block)
# Climate compensation has NO day-5 entry yet (deterministic 1→6 order, comes after Carbon tax).
check('Day 5: today-so-far excludes Climate compensation (not yet answered today)',
      'Climate compensation' not in today_block)
# Reasoning ("(Why: ...)") should appear in today-so-far because fixture now seeds Day-5
# reasoning for policies 1-4.
check('Day 5: today-so-far entries include "(Why:" clauses (reasoning attached)',
      '(Why:' in today_block)

# (8) Day 0: no PAST-day sections (summaries / reflections / own-reasoning).
# Today-so-far IS expected to fire at Day 0 in package mode.
past_day_headers = [
    'Summary of recent days:',
    'Recent reflections following received messages:',
    'Your considered position in recent days:',
]
check('Day 0: no past-day headers (summaries/reflections/own-reasoning)',
      not any(h in ctx0 for h in past_day_headers))
check('Day 0: today-so-far section fires (package mode, other policies answered today)',
      "Your answers so far in today's survey:" in ctx0)
# Day-0 today-so-far now also carries reasoning (fixture seeds Day-0 reasoning for all policies).
check('Day 0: today-so-far entries include "(Why:" clauses (reasoning attached)',
      '(Why:' in ctx0)

# (9) Day 2: daily summaries still absent (1..d-2 = 1..0 is empty at Day 2).
check('Day 2: no daily-summaries section', 'Summary of recent days:' not in ctx2)
check('Day 2: anchor section present', 'Original prior position on' in ctx2)
# Vivid reflections AT Day 2 now fire because fixture seeds Day-1 and Day-2 reflections.
check('Day 2: vivid reflections section fires (Day 1 and 2 entries in fixture)',
      'Recent reflections following received messages:' in ctx2)
# Today-so-far at Day 2 now carries reasoning (fixture seeds Day-2 reasoning for all policies).
check('Day 2: today-so-far entries include "(Why:" clauses (reasoning attached)',
      '(Why:' in ctx2)
# Day 2 considered-position section fires for Day-1 Carbon-tax reasoning (target-scoped).
check('Day 2: considered-position section fires (Day-1, Day-2 Carbon-tax reasoning)',
      'Your considered position in recent days:' in ctx2)

# Print results.
n_pass = sum(1 for _, ok in checks if ok)
print(f'\n{n_pass}/{len(checks)} structural checks passed.\n')
for label, ok in checks:
    marker = 'OK ' if ok else 'FAIL'
    print(f'  [{marker}] {label}')

assert all(ok for _, ok in checks), 'One or more structural checks failed — fix before porting to src.'



46/46 structural checks passed.

  [OK ] Day 0: no "Remember who you are" line
  [OK ] Day 2: no "Remember who you are" line
  [OK ] Day 5: no "Remember who you are" line
  [OK ] Day 2: carbon-tax anchor present
  [OK ] Day 5: carbon-tax anchor present
  [OK ] Day 2: green-housing anchor NOT in context (target-scoped)
  [OK ] Day 2: petrol-car anchor NOT in context (target-scoped)
  [OK ] Day 5: green-housing anchor NOT in context (target-scoped)
  [OK ] Day 5: petrol-car anchor NOT in context (target-scoped)
  [OK ] Day 5: all 5 named headers present
  [OK ] Day 5: header order is anchor < summaries < reflections < own-reasoning < today
  [OK ] Day 5: daily summary for day 1 included
  [OK ] Day 5: daily summary for day 2 included
  [OK ] Day 5: daily summary for day 3 included
  [OK ] Day 5: no day-4 daily summary (vivid window keeps it verbatim)
  [OK ] Day 5: vivid reflections include day 4
  [OK ] Day 5: vivid reflections include day 5
  [OK ] Day 5: vivid reflections exclude day

## 6. Single-policy mode sanity check

Same assembly but with `policy_id=3` (single-policy mode, ban petrol cars).
The within-day section must NOT fire (single-policy means there are no
"other policies answered today"). The anchor still scopes to `target_policy_id`.

In [8]:
# Build a tiny single-policy variant: only one policy, no today-so-far ever.
single_agent = SimpleNamespace(
    persona_text=agent.persona_text,
    day0_anchors={3: agent.day0_anchors[3]},
    daily_summaries={
        (1, 3): 'Today the pro-side argued for the 2030 deadline; I held my position because '
                'the infrastructure case was not addressed.',
        (2, 3): 'A peer brought up the rural-transport angle which strengthened my hesitation.',
    },
    reflections=[r for r in agent.reflections if r['day'] in (3, 4)],
    survey_reasoning={3: agent.survey_reasoning[3]},
    opinion_history={3: agent.opinion_history[3]},
)

# For single-policy mode, policy_id IS the target. target_policy_id matches.
ctx_sp = assemble_context_v2(single_agent, day=4, policy_id=3, target_policy_id=3)
print(banner('SINGLE-POLICY context  (target = policy 3, ban petrol cars)'))
print(ctx_sp)
print(f'\n[len: {len(ctx_sp)} chars]')

assert "Your answers so far in today's survey:" not in ctx_sp, \
    'Within-day section should not appear in single-policy mode'
assert 'Original prior position on "Ban new petrol cars by 2030":' in ctx_sp
assert 'Remember who you are' not in ctx_sp
print('Single-policy mode sanity OK.')


SINGLE-POLICY context  (target = policy 3, ban petrol cars)

Demographically, I am a 47-year-old female living in the North West, United Kingdom. My ethnic background is White, and I hold an undergraduate degree. Financially, my gross household income falls into the 30–50k bracket. I am a parent. Politically, I position myself slightly left of centre. In the 2019 General Election I voted Labour; in the EU Referendum I voted Remain.

When it comes to my core values and worldview: I strongly value self-transcendence (care for others and the environment). I am moderately open to new ideas. I score low on social-dominance orientation and right-wing authoritarianism. I am moderately concerned about traditional norms.

Original prior position on "Ban new petrol cars by 2030":
I slightly oppose banning new petrol cars by 2030. The principle is right but I don't see the charging infrastructure or used-EV market being ready that fast for lower-income households.

Recent reflections following r

## 7. Summary

If section 4 reads sensibly, section 5 reports all checks passing, and
section 6 doesn't accidentally emit a today-so-far block, the v2 assembly
logic is ready to port to `src/cag/abm/agent.py`. Implementation map per
`/memories/session/plan.md`:

- `assemble_context(day, policy_id, target_policy_id)` — replace existing
  body with the section-builder pattern above; drop trailing demographic
  reminder.
- New `day0_anchors: dict[policy_id, str]` agent attribute, initialised
  empty in `__init__`.
- New `compress_day0_anchor(policy_id, ...)` helper (LLM call) populated
  once at end of Day 0 by `_run_one_day` in `src/cag/abm/sim.py`.
- Rewrite `compress_daily_memory` to unify reflections + own survey
  reasoning into the 6–8 sentence package-scoped summary.
- `administer_survey` takes new `target_policy_id` kwarg, forwards to
  `get_system_prompt` → `assemble_context`.
- `run_end_of_day_survey` plumbs `target_policy_id` through.
- `sim.py` `_run_one_day` blind-spot fix: move `manage_memory` ABOVE EOD
  survey loop.
- Test updates in `tests/test_memory.py` (rewrite
  `test_persona_included_at_end`, add 5 new tests).